# Prep one RFROMV2.3 netCDF for NODD and upload it

This notebook prepares **one** output netCDF for the NOAA Open Data Dissemination (NODD)
GCP bucket and uploads it, so the workflow can be tested before scaling to all files
(GitHub issue #1).

**What it does**

1. Build the list of output files ("blocks" of 100 time steps) and cross-walk each block
   to the monthly source files on ERDDAP.
2. Download the monthly source netCDFs for one block from ERDDAP to a scratch directory.
3. Open + combine them and select exactly the block's 100 time steps.
4. Fix / update metadata.
5. Rechunk to `(time=100, mean_pressure=1, latitude=180, longitude=180)` (~13 MB per chunk).
6. Write the output netCDF locally (~24 GB).
7. Upload that single file to the NODD bucket.

**Source dataset** (ERDDAP griddap id `argo_rfromv23_temp`): `ocean_temperature(time, mean_pressure, latitude, longitude)`,
float32, dims `time=1670, mean_pressure=58, latitude=720, longitude=1440`. Monthly source files hold
4–5 seven-day time steps and are unchunked.

> **Disk needed:** one block downloads ~23–25 monthly files (~1.2 GB each ≈ 30 GB) plus the
> ~24 GB output. Make sure the scratch directory has ~55 GB free. A local disk (or S3 scratch)
> is fine; the monthly downloads can be deleted afterward (last cell).

## Setup

In [ ]:
import os
import shutil

import numpy as np
import pandas as pd
import requests
import xarray as xr
import gcsfs

## Configuration

Everything that changes between variables (temp vs. sal) or between test/production lives here.

In [ ]:
# --- Source (ERDDAP) -------------------------------------------------------
DATASET_ID = "argo_rfromv23_temp"          # ERDDAP griddap dataset id
DATA_VAR   = "ocean_temperature"            # main data variable in the files
MONTHLY_PREFIX = "RFROMV23_TEMP_STABLE"     # monthly source file prefix on ERDDAP

FILES_URL = f"https://data.pmel.noaa.gov/pmel/erddap/files/{DATASET_ID}"
TIME_CSV  = f"https://data.pmel.noaa.gov/pmel/erddap/griddap/{DATASET_ID}.csv?time"

# --- Output layout ---------------------------------------------------------
OUT_PREFIX = "RFROMV23_TEMP_STABLE"         # output file prefix (NODD naming)
BLOCK_SIZE = 100                            # time steps per output file
BLOCK_INDEX = 0                             # <-- which block to process/upload in this test

# Physical chunk sizes, in variable dim order (time, mean_pressure, latitude, longitude).
# 100 * 1 * 180 * 180 * 4 bytes = ~12.96 MB per chunk.
CHUNKS = {"time": 100, "mean_pressure": 1, "latitude": 180, "longitude": 180}

# --- Scratch / local paths -------------------------------------------------
SCRATCH_DIR = "/home/jovyan/shared-public/rfromv-scratch"   # download + output dir
DOWNLOAD_DIR = os.path.join(SCRATCH_DIR, "erddap_monthly")
OUTPUT_DIR   = os.path.join(SCRATCH_DIR, "nodd")
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- NODD (GCP) destination ------------------------------------------------
GCS_TOKEN = "/home/jovyan/.config/gcloud/application_default_credentials.json"
NODD_BUCKET = "noaa-oar-rfrom"
NODD_NETCDF_DIR = "netcdfs"                 # <-- confirm the exact dir name in the bucket
NODD_DEST = f"gs://{NODD_BUCKET}/{NODD_NETCDF_DIR}"

## Step 1 — Build the file blocks

Read the full time axis from ERDDAP, split it into blocks of `BLOCK_SIZE` time steps, and
cross-walk each block to the monthly source files that cover it. The output filename encodes
the first and last date in the block.

In [ ]:
def make_file_blocks(times, block_size=BLOCK_SIZE):
    """Split the time index into blocks and map each to its monthly source files."""
    times = pd.DatetimeIndex(times)
    blocks = []
    for i in range(0, len(times), block_size):
        block_times = times[i:i + block_size]
        months = block_times.to_period("M").unique()
        urls = [
            f"{FILES_URL}/{MONTHLY_PREFIX}_{m.year}_{m.month:02d}.nc"
            for m in months
        ]
        start, end = block_times[0], block_times[-1]
        filename = f"{OUT_PREFIX}_{start:%Y-%m-%d}_{end:%Y-%m-%d}.nc"
        blocks.append({
            "block": i // block_size,
            "start": start,
            "end": end,
            "n_times": len(block_times),
            "filename": filename,
            "urls": urls,
        })
    return blocks


# Read the time axis (row 0 is the header, row 1 is the units row -> skip it)
times = pd.read_csv(TIME_CSV, skiprows=[1])
times["time"] = pd.to_datetime(times["time"])
times = pd.DatetimeIndex(times["time"])

blocks = make_file_blocks(times)
print(f"{len(times)} time steps -> {len(blocks)} output files\n")
for b in blocks:
    print(f"[{b['block']:2d}] {b['filename']}  ({b['n_times']} steps, {len(b['urls'])} monthly files)")

In [ ]:
# The single block we will process and upload in this test
block = blocks[BLOCK_INDEX]
print("Processing block", block["block"])
print("Output file:", block["filename"])
print("Time range :", block["start"].date(), "->", block["end"].date(), f"({block['n_times']} steps)")
print("Source files:")
for u in block["urls"]:
    print("  ", u.split("/")[-1])

## Step 2 — Download the monthly source files for this block

Stream each monthly file to `DOWNLOAD_DIR`. Files already present with the right size are
skipped, so the cell is safe to re-run.

In [ ]:
def download(url, dest, chunk=16 * 1024 * 1024):
    """Download url -> dest, skipping if a complete copy already exists."""
    head = requests.head(url, timeout=60)
    head.raise_for_status()
    remote_size = int(head.headers.get("Content-Length", 0))
    if os.path.exists(dest) and remote_size and os.path.getsize(dest) == remote_size:
        print(f"  skip (have) {os.path.basename(dest)}")
        return dest
    tmp = dest + ".part"
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for block_bytes in r.iter_content(chunk_size=chunk):
                f.write(block_bytes)
    os.replace(tmp, dest)
    print(f"  got  {os.path.basename(dest)}  ({os.path.getsize(dest) / 1e9:.2f} GB)")
    return dest


local_files = []
for url in block["urls"]:
    dest = os.path.join(DOWNLOAD_DIR, url.split("/")[-1])
    local_files.append(download(url, dest))

total_gb = sum(os.path.getsize(f) for f in local_files) / 1e9
print(f"\nDownloaded {len(local_files)} files, {total_gb:.1f} GB total")

## Step 3 — Open, combine, and select the block's time steps

The monthly files at the block boundaries can contain a few time steps outside the block, so
we slice to exactly `[start, end]` — this returns exactly the block's time steps.

In [ ]:
ds = xr.open_mfdataset(
    local_files,
    engine="h5netcdf",
    combine="by_coords",   # order by the time coordinate in each file
    chunks={},             # lazy; we rechunk explicitly in Step 5
)

# Trim to exactly this block's time steps
ds = ds.sel(time=slice(block["start"], block["end"]))

assert ds.sizes["time"] == block["n_times"], (
    f"expected {block['n_times']} times, got {ds.sizes['time']}"
)
print("Selected dims:", dict(ds.sizes))
ds

## Step 4 — Fix / update metadata

Preserve the source global attributes and add a short processing note to `history`. Adjust
here if other attributes need correcting.

In [ ]:
stamp = pd.Timestamp.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
note = (
    f"{stamp}: repackaged for NODD from ERDDAP {DATASET_ID} monthly files; "
    f"rechunked to {tuple(CHUNKS[d] for d in ('time','mean_pressure','latitude','longitude'))} "
    f"(time, mean_pressure, latitude, longitude)."
)
ds.attrs["history"] = note + "\n" + ds.attrs.get("history", "")

# Sanity-check the key attributes
for k in ("title", "institution", "source", "references", "license"):
    print(f"{k}: {ds.attrs.get(k, '<missing>')}")

## Step 5 — Rechunk and set the on-disk encoding

Dask chunks (for streaming the write) and the netCDF `chunksizes` encoding are both set to
`CHUNKS`. The time chunk is capped at the block length so the last (short) block stays valid.

In [ ]:
# time chunk cannot exceed the number of time steps in this block
tchunk = min(CHUNKS["time"], ds.sizes["time"])
dask_chunks = {**CHUNKS, "time": tchunk}
ds = ds.chunk(dask_chunks)

# netCDF chunksizes must be given in the variable's dimension order
var_dims = ds[DATA_VAR].dims
chunksizes = tuple(dask_chunks[d] for d in var_dims)

encoding = {
    DATA_VAR: {
        "chunksizes": chunksizes,
        "dtype": "float32",
        # Optional lossless compression (smaller files, slower write). Off by default so
        # the output size matches the ~24 GB estimate; enable if NODD storage is a concern:
        # "zlib": True, "complevel": 1,
    },
    # Preserve the original time encoding
    "time": {"units": "seconds since 1970-01-01T00:00:00Z", "dtype": "float64"},
}
print("variable dims :", var_dims)
print("chunksizes    :", chunksizes)

## Step 6 — Write the output netCDF locally

In [ ]:
out_path = os.path.join(OUTPUT_DIR, block["filename"])
print("Writing", out_path, "...")
ds.to_netcdf(out_path, engine="h5netcdf", encoding=encoding, mode="w")
print(f"Done: {os.path.getsize(out_path) / 1e9:.2f} GB")

In [ ]:
# Verify the written file: chunking and shape
check = xr.open_dataset(out_path, engine="h5netcdf")
print("dims           :", dict(check.sizes))
print("stored chunking:", check[DATA_VAR].encoding.get("chunksizes"))
check.close()

## Step 7 — Upload the single file to NODD

Uses `gcsfs` with the application-default credentials (same approach that already works for
the `index.html` upload).

In [ ]:
fs = gcsfs.GCSFileSystem(token=GCS_TOKEN)

dest = f"{NODD_DEST}/{block['filename']}"
print(f"Uploading {out_path}\n       -> {dest}")
fs.put(out_path, dest)
print("Uploaded.")

In [ ]:
# Verify it landed
info = fs.info(dest)
print(dest.replace("gs://", ""))
print(f"  size: {info['size'] / 1e9:.2f} GB")

## Step 8 — (optional) Clean up the downloaded monthly files

The monthly source files are only needed to build the output. Remove them once the upload is
verified to free scratch space.

In [ ]:
# for f in local_files:
#     os.remove(f)
# print("Removed", len(local_files), "monthly source files")